# 15.6 - Experiment Tracking

Status: VERIFIED

## What Are We Solving?
Without tracking, you cannot remember which configuration produced which result. Experiment tracking turns chaos into a searchable history.

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
import time
import json
from pathlib import Path
print("All imports OK")

All imports OK


## Manual Experiment Tracker

In [2]:
class ExperimentTracker:
    def __init__(self):
        self.experiments = []
    
    def log_experiment(self, name: str, params: dict, metrics: dict, tags: list = None):
        exp = {
            "name": name,
            "params": params,
            "metrics": metrics,
            "tags": tags or [],
            "timestamp": time.time(),
        }
        self.experiments.append(exp)
        print(f"Logged: {name} | Metrics: {metrics}")
    
    def compare(self, metric_name: str) -> list:
        return sorted(self.experiments, key=lambda e: e["metrics"].get(metric_name, 0), reverse=True)
    
    def best(self, metric_name: str) -> dict:
        ranked = self.compare(metric_name)
        return ranked[0] if ranked else None
    
    def summary(self) -> dict:
        return {
            "total_experiments": len(self.experiments),
            "experiments": [{"name": e["name"], "metrics": e["metrics"]} for e in self.experiments],
        }

tracker = ExperimentTracker()

# Log experiments
tracker.log_experiment("baseline_logreg", {"model": "logreg", "lr": 0.01}, {"f1": 0.72, "accuracy": 0.85})
tracker.log_experiment("rf_100", {"model": "rf", "n_estimators": 100}, {"f1": 0.81, "accuracy": 0.89})
tracker.log_experiment("rf_200", {"model": "rf", "n_estimators": 200}, {"f1": 0.83, "accuracy": 0.90})
tracker.log_experiment("xgb_tuned", {"model": "xgb", "lr": 0.1, "depth": 6}, {"f1": 0.85, "accuracy": 0.91})

best = tracker.best("f1")
print(f"\nBest experiment: {best['name']} with F1={best['metrics']['f1']}")

Logged: baseline_logreg | Metrics: {'f1': 0.72, 'accuracy': 0.85}
Logged: rf_100 | Metrics: {'f1': 0.81, 'accuracy': 0.89}
Logged: rf_200 | Metrics: {'f1': 0.83, 'accuracy': 0.9}
Logged: xgb_tuned | Metrics: {'f1': 0.85, 'accuracy': 0.91}

Best experiment: xgb_tuned with F1=0.85


## Real Experiment with sklearn

In [3]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Generate dataset
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Run real experiments
configs = [
    {"name": "logreg_c1", "model": LogisticRegression(C=1.0, random_state=42, max_iter=1000)},
    {"name": "logreg_c01", "model": LogisticRegression(C=0.1, random_state=42, max_iter=1000)},
    {"name": "rf_100", "model": RandomForestClassifier(n_estimators=100, random_state=42)},
    {"name": "rf_200", "model": RandomForestClassifier(n_estimators=200, random_state=42)},
    {"name": "gbdt_100", "model": GradientBoostingClassifier(n_estimators=100, random_state=42)},
]

tracker2 = ExperimentTracker()
for config in configs:
    config["model"].fit(X_train, y_train)
    y_pred = config["model"].predict(X_test)
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    tracker2.log_experiment(config["name"], {"model": type(config["model"]).__name__}, {"f1": round(f1, 4), "accuracy": round(acc, 4)})

print(f"\nBest by F1: {tracker2.best('f1')['name']}")

Logged: logreg_c1 | Metrics: {'f1': 0.8557, 'accuracy': 0.855}
Logged: logreg_c01 | Metrics: {'f1': 0.8657, 'accuracy': 0.865}


Logged: rf_100 | Metrics: {'f1': 0.902, 'accuracy': 0.9}


Logged: rf_200 | Metrics: {'f1': 0.8911, 'accuracy': 0.89}


Logged: gbdt_100 | Metrics: {'f1': 0.9109, 'accuracy': 0.91}

Best by F1: gbdt_100


## Experiment Comparison Visualization

In [4]:
# Compare experiments
summary = tracker2.summary()
names = [e["name"] for e in summary["experiments"]]
f1s = [e["metrics"]["f1"] for e in summary["experiments"]]
accs = [e["metrics"]["accuracy"] for e in summary["experiments"]]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].barh(names, f1s, color='steelblue')
axes[0].set_xlabel('F1 Score')
axes[0].set_title('Experiment Comparison (F1)')
axes[0].grid(True, alpha=0.3)

axes[1].barh(names, accs, color='seagreen')
axes[1].set_xlabel('Accuracy')
axes[1].set_title('Experiment Comparison (Accuracy)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('experiment_tracking.png', dpi=100, bbox_inches='tight')
plt.show()
print("Experiment comparison saved")

Experiment comparison saved


## Reproducibility Checklist

In [5]:
# Reproducibility checklist
def check_reproducibility(experiment: dict) -> dict:
    checks = {
        "has_name": bool(experiment.get("name")),
        "has_params": bool(experiment.get("params")),
        "has_metrics": bool(experiment.get("metrics")),
        "has_timestamp": "timestamp" in experiment,
        "params_logged": len(experiment.get("params", {})) > 0,
    }
    checks["all_passed"] = all(checks.values())
    return checks

# Check our experiments
for exp in tracker2.experiments[:2]:
    checks = check_reproducibility(exp)
    print(f"{exp['name']}: {'PASS' if checks['all_passed'] else 'FAIL'}")

logreg_c1: PASS
logreg_c01: PASS


In [6]:
# Verification
assert tracker2.best("f1") is not None, "Must have a best experiment"
assert len(tracker2.experiments) == 5, "Must have 5 experiments"
print("VERIFICATION PASSED: Phase 15.6 complete")

VERIFICATION PASSED: Phase 15.6 complete
